# NICE DCV

This Jupyter notebook will allow you to install NICE DCV to the desired VMs
https://docs.aws.amazon.com/dcv/

Rocky Linux 8,9 and Ubuntu 20, 22 are supported.
You need to input the name of the slice and the name of the VM(s) to install NICE DCV.


## Step 1: Import the FABlib Library


In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
                     
fablib.show_config()

## Step 2: Check your existing slices



In [ ]:
try:
    for slice in fablib.get_slices():
        print(f"{slice}")
except Exception as e:
    print(f"Exception: {e}")

## Step 3: Observe the Slice's Attributes

### Select the slice 

In [ ]:
slice_name=f"slice"

In [ ]:
try:
    slice = fablib.get_slice(name=slice_name)
    print(f"{slice}")
except Exception as e:
    print(f"Exception: {e}")

### Print the Node List

In [ ]:
try:
    slice = fablib.get_slice(name=slice_name)

    print(f"{slice.list_nodes()}")
except Exception as e:
    print(f"Exception: {e}")

### Print the Node Details (Optional)

In [ ]:
#try:
#    slice = fablib.get_slice(name=slice_name)
#    for node in slice.get_nodes():
#        print(f"{node}")
#except Exception as e:
#    print(f"Exception: {e}")

### Print the Interfaces (Optional)

In [ ]:
#try:
#    slice = fablib.get_slice(name=slice_name)
#    print(f"{slice.list_interfaces()}")
#except Exception as e:
#    print(f"Exception: {e}")

## Step 4: Install NICE DCV


### Select the node to install NICE DCV

In [ ]:
node_name="s1-br"

### Upload scripts to the node

In [ ]:
node = slice.get_node(name=node_name)

try:
    node.upload_file('../../tools/node_tools/enable_dcv_pre_1.sh', 'enable_dcv_pre_1.sh')
    node.upload_file('../../tools/node_tools/enable_dcv_pre_2.sh', 'enable_dcv_pre_2.sh')
    node.upload_file('../../tools/node_tools/enable_dcv.sh', 'enable_dcv.sh')
except Exception as e:
    print(f"Exception: {e}")

### Install

In [ ]:
try:
    stdout, stderr = node.execute(f'chmod +x enable_dcv_pre_1.sh && sudo ./enable_dcv_pre_1.sh > /tmp/dcv_install_pre_1.log 2>&1')
except Exception as e:
    print(f"Exception: {e}")

In [ ]:
reboot = 'sudo reboot'
try:
    print(reboot)
    node.execute(reboot)
    
    slice.wait_ssh(timeout=360,interval=10,progress=True)

    print("Now testing SSH abilites to reconnect...",end="")
    slice.update()
    slice.test_ssh()
    print("Reconnected!")

except Exception as e:
    print(f"Fail: {e}")  

node.config()
print("Done")    

In [ ]:
try:
    stdout, stderr = node.execute(f'chmod +x enable_dcv_pre_2.sh && sudo ./enable_dcv_pre_2.sh > /tmp/dcv_install_pre_2.log 2>&1')
except Exception as e:
    print(f"Exception: {e}")

In [ ]:
reboot = 'sudo reboot'
try:
    print(reboot)
    node.execute(reboot)
    
    slice.wait_ssh(timeout=360,interval=10,progress=True)

    print("Now testing SSH abilites to reconnect...",end="")
    slice.update()
    slice.test_ssh()
    print("Reconnected!")

except Exception as e:
    print(f"Fail: {e}")  

node.config()
print("Done")    

In [ ]:
try:
    stdout, stderr = node.execute(f'chmod +x enable_dcv.sh && sudo ./enable_dcv.sh > /tmp/dcv_install.log 2>&1')
except Exception as e:
    print(f"Exception: {e}")

In [ ]:
command = "DCV_CURRENT=$(sudo dcv list-sessions -j | jq '.[].id'); [[ -z ${DCV_CURRENT} ]] && sudo dcv create-session --type virtual --owner `whoami` session1 || dcv list-sessions"

try:
    node = slice.get_node(name=node_name)  
    stdout, stderr = node.execute(command)
    print(f"stdout: {stdout}")
except Exception as e:
    print(f"Exception: {e}")

In [ ]:
# Generated password for the user "rocky or ubuntu"

command = "[ -f /tmp/password.user ] && cat /tmp/password.user || echo File missing"

try:
    node = slice.get_node(name=node_name)  
    stdout, stderr = node.execute(command)
    print(f"stdout: {stdout}")
except Exception as e:
    print(f"Exception: {e}")

***SSH to the VMs***

```
ssh -X -F ~/.ssh/config_fabric_exp -i ~/.ssh/mcevik_fabric_sliver_1 -L 9443:localhost:8443 rocky@<IP_Address>
```

On the web-browser
https://localhost:9443